# Сбор демонстраций с клавиатуры

Сбор демонстраций в заданной среде.
Задача — взять чашку и поставить её на тарелку. Среда засчитывает успех, когда чашка стоит на тарелке, захват разомкнут, а эффектор находится над чашкой.

<img src="./media/teleop.gif" width="480" height="360">

Клавиши: WASD — перемещение в плоскости x-y, R/F — ось z, Q/E — наклон, стрелки — остальные повороты.

Пробел переключает состояние захвата (открыт/закрыт), клавиша Z сбрасывает среду и отбрасывает кэш текущего эпизода.

Пояснения к наложению изображений:
- Вверху справа: вид агента
- Внизу справа: вид с запястья (от первого лица)
- Вверху слева: боковой вид
- Внизу слева: вид сверху


Старайтесь собирать данные аккуратнее, чем показано в видео: точность важнее скорости.

In [ ]:
# pip install -r requirements.txt
# Рекомендуется окружение python310

In [1]:
import os
os.environ["GRIPPER_RATE_PER_STEP"] = "0.05"
os.environ["SUCCESS_Z_THRESHOLD"] = "0.35"
import env_config

[Env Info] 运行环境已就绪，当前工作目录: C:\\Users\\<пользователь>\\lerobot-mujoco-tutorial


In [2]:
import sys
import random
import numpy as np
import os
from PIL import Image
from mujoco_env.y_env import SimpleEnv
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

In [3]:
# If you want to randomize the object positions, set this to None
# If you fix the seed, the object positions will be the same every time
SEED = 0 
# SEED = None <- Uncomment this line to randomize the object positions

REPO_NAME = 'datawhale_eai_nova5_pnp' #pick-and-place
NUM_DEMO = 1 # Number of demonstrations to collect
ROOT = "./demo_data" # The root directory to save the demonstrations

Не забудьте распаковать эту папку

![fig1](./media/fig1.png)

GRIPPER_RATE_PER_STEP (скорость изменения команды захвата за шаг, по умолчанию 0.03)
0.02 ~ 0.05 работает стабильно; при значениях > 0.08 чаще возникают рывки и проникновения в объекты, при <= 0 почти не двигается (не рекомендуется)


SUCCESS_XY_THRESHOLD (по умолчанию 0.10)

SUCCESS_Z_THRESHOLD (по умолчанию 0.12)

SUCCESS_EE_HEIGHT_THRESHOLD (для Nova5 по умолчанию 0.95)

SUCCESS_REQUIRE_GRIPPER_OPEN (по умолчанию 1, нужно отпустить захват)

SUCCESS_GRIPPER_OPEN_NORM_THRESHOLD (по умолчанию 0.30)

Ключевой момент: эти переменные нужно задать ДО `PnPEnv = SimpleEnv(...)`; изменения в середине не повлияют на уже созданные объекты.
Нужно в заголовке notebook


In [4]:
TASK_NAME = 'Put mug cup on the plate' 
xml_path = './asset/example_scene_nova5.xml'
# xml_path = 'asset/example_scene_nova5.xml'
# Define the environment
PnPEnv = SimpleEnv(xml_path, seed = SEED, state_type = 'joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[26] n_qvel:[24] n_qacc:[24] n_ctrl:[7]
 integrator:[RK4]

n_body:[23]
 [0/23] [world] mass:[0.00]kg
 [1/23] [front_object_table] mass:[1.00]kg
 [2/23] [camera] mass:[0.00]kg
 [3/23] [camera2] mass:[0.00]kg
 [4/23] [camera3] mass:[0.00]kg
 [5/23] [nova5_mount] mass:[1.67]kg
 [6/23] [Link1] mass:[1.08]kg
 [7/23] [Link2] mass:[2.55]kg
 [8/23] [Link3] mass:[1.41]kg
 [9/23] [Link4] mass:[0.44]kg
 [10/23] [Link5] mass:[0.57]kg
 [11/23] [Link6] mass:[0.44]kg
 [12/23] [camera_wrist] mass:[0.00]kg
 [13/23] [left_outer_knuckle] mass:[0.03]kg
 [14/23] [left_inner_knuckle] mass:[0.01]kg
 [15/23] [left_inner_finger] mass:[0.01]kg
 [16/23] [right_inner_knuckle] mass:[0.01]kg
 [17/23] [right_inner_finger] mass:[0.01]kg
 [18/23] [right_outer_knuckle] mass:[0.03]kg
 [19/23] [body_obj_mug_5] mass:[0.00]kg
 [20/23] [object_mug_5] mass:[0.08]kg
 [21/23] [body_obj_plate_11] mass:[0.00

## Определение признаков набора данных и создание своего набора
Структура набора данных выглядит так:
```
fps = 20,
features={
    "observation.image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.wrist_image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channel"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["state"], # x, y, z, roll, pitch, yaw
    },
    "action": {
        "dtype": "float32",
        "shape": (7,),
        "names": ["action"], # 6 joint angles and 1 gripper
    },
    "obj_init": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["obj_init"], # just the initial position of the object. Not used in training.
    },
},
```

Данные будут сохранены в каталог `./demo_data`.


# Горячая перезагрузка окружения (если содержимое пакета изменилось, можно сделать так.)

In [16]:
import importlib

# 1) Горячая загрузка модулей (сначала перезагрузить зависимости, затем главный модуль)
import mujoco_env.ik as ik_mod
import mujoco_env.mujoco_parser as parser_mod
import mujoco_env.y_env as y_env_mod

importlib.reload(ik_mod)
importlib.reload(parser_mod)
importlib.reload(y_env_mod)

# 2) Закрываем viewer старого окружения (если он существует)
try:
    PnPEnv.env.close_viewer()
except Exception:
    pass

# 3) Заново привязываем класс и создаём новое окружение
SimpleEnv = y_env_mod.SimpleEnv
PnPEnv = SimpleEnv('./asset/example_scene_nova5.xml', seed=SEED, state_type='joint_angle')

print("hot reload done:", PnPEnv.ee_body_name)
print(PnPEnv.env.cam_names)
print("agent:", PnPEnv.agent_cam_name, "wrist:", PnPEnv.wrist_cam_name)
print("wrist_cam:", PnPEnv.wrist_cam_name)
print("n_ctrl:", PnPEnv.env.n_ctrl)


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[26] n_qvel:[24] n_qacc:[24] n_ctrl:[7]
 integrator:[RK4]

n_body:[23]
 [0/23] [world] mass:[0.00]kg
 [1/23] [front_object_table] mass:[1.00]kg
 [2/23] [camera] mass:[0.00]kg
 [3/23] [camera2] mass:[0.00]kg
 [4/23] [camera3] mass:[0.00]kg
 [5/23] [nova5_mount] mass:[1.67]kg
 [6/23] [Link1] mass:[1.08]kg
 [7/23] [Link2] mass:[2.55]kg
 [8/23] [Link3] mass:[1.41]kg
 [9/23] [Link4] mass:[0.44]kg
 [10/23] [Link5] mass:[0.57]kg
 [11/23] [Link6] mass:[0.44]kg
 [12/23] [camera_wrist] mass:[0.00]kg
 [13/23] [left_outer_knuckle] mass:[0.03]kg
 [14/23] [left_inner_knuckle] mass:[0.01]kg
 [15/23] [left_inner_finger] mass:[0.01]kg
 [16/23] [right_inner_knuckle] mass:[0.01]kg
 [17/23] [right_inner_finger] mass:[0.01]kg
 [18/23] [right_outer_knuckle] mass:[0.03]kg
 [19/23] [body_obj_mug_5] mass:[0.00]kg
 [20/23] [object_mug_5] mass:[0.08]kg
 [21/23] [body_obj_plate_11] mass:[0.00

In [5]:
create_new = True
if os.path.exists(ROOT):
    print(f"Directory {ROOT} already exists.")
    ans = input("Do you want to delete it? (y/n) ")
    if ans == 'y':
        import shutil
        shutil.rmtree(ROOT)
    else:
        create_new = False


if create_new:
    dataset = LeRobotDataset.create(
                repo_id=REPO_NAME,
                root = ROOT, 
                robot_type="nova5",
                fps=20, # 20 frames per second
                features={
                    "observation.image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channels"],
                    },
                    "observation.wrist_image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channel"],
                    },
                    "observation.state": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["state"], # x, y, z, roll, pitch, yaw
                    },
                    "action": {
                        "dtype": "float32",
                        "shape": (7,),
                        "names": ["action"], # 6 joint angles and 1 gripper
                    },
                    "obj_init": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["obj_init"], # just the initial position of the object. Not used in training.
                    },
                },
                image_writer_threads=10,
                image_writer_processes=5,
        )
else:
    print("Load from previous dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)

Directory ./demo_data already exists.


Do you want to delete it? (y/n)  y


## Управление с клавиатуры
Можно телеуправлять манипулятором с клавиатуры и собирать данные
```
---------     -----------------------
   w       ->        backward
s  a  d        left   forward   right
---------      -----------------------
In x, y plane

---------
R: Moving Up
F: Moving Down
---------
In z axis

---------
Q: Tilt left
E: Tilt right
UP: Look Upward
Down: Look Donward
Right: Turn right
Left: Turn left
---------
For rotation

---------
SPACEBAR: Toggle Gripper
--------

---------
z: reset
--------
```
Сброс среды удаляет кэш текущего демонстрационного эпизода и начинает сбор заново.


Подробнее о части Rotation (вращение):
В управлении роботом эта часть обычно соответствует эйлеровым углам (Euler Angles) в трёхмерном пространстве:

Tilt (Q/E): соответствует Roll (крен). Представьте, что конец манипулятора наклоняется влево-вправо, как при закручивании винта.

Look Upward/Downward (UP/Down): соответствует Pitch (тангаж). Управляет поднятием или опусканием «головы» конца манипулятора.

Turn Left/Right (Left/Right): соответствует Yaw (рысканье). Управляет поворотом конца манипулятора влево и вправо в горизонтальной плоскости.

### Начинаем телеуправление и сбор данных

**Чтобы получить сигнал успеха, разожмите захват и поднимите эффектор над чашкой.**


Центры чашки и тарелки по XY должны быть достаточно близко (< 0.1)

Захват должен быть действительно открыт (rh_r1 < 0.1; это легко заедает)

Эффектор (tcp_link) нужно поднять до z > 0.9

Только когда все условия выполнены, срабатывает done=True

In [ ]:
action = np.zeros(7)
episode_id = 0
record_flag = False # Start recording when the robot starts moving
while PnPEnv.env.is_viewer_alive() and episode_id < NUM_DEMO:
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # check if the episode is done
        done = PnPEnv.check_success()
        if done and record_flag:
            # Save the episode data and reset the environment
            dataset.save_episode()
            episode_id += 1
            record_flag = False
            if episode_id >= NUM_DEMO:
                break
            PnPEnv.reset(seed=SEED)
            continue
        # Teleoperate the robot and get delta end-effector pose with gripper
        action, reset  = PnPEnv.teleop_robot()
        if not record_flag and np.any(action != 0):
            record_flag = True
            print("Start recording")
        if reset:
            # Reset the environment and clear the episode buffer
            # This can be done by pressing 'z' key
            PnPEnv.reset(seed=SEED)
            # PnPEnv.reset()
            dataset.clear_episode_buffer()
            record_flag = False
            continue
        # Step the environment
        # Get the end-effector pose and images
        ee_pose = PnPEnv.get_ee_pose()
        agent_image,wrist_image = PnPEnv.grab_image()
        # # resize to 256x256
        agent_image = Image.fromarray(agent_image)
        wrist_image = Image.fromarray(wrist_image)
        agent_image = agent_image.resize((256, 256))
        wrist_image = wrist_image.resize((256, 256))
        agent_image = np.array(agent_image)
        wrist_image = np.array(wrist_image)
        PnPEnv.step(action)
        joint_action = PnPEnv.get_commanded_joint_action()
        if record_flag:
            # Add the frame to the dataset
            dataset.add_frame( {
                    "observation.image": agent_image,
                    "observation.wrist_image": wrist_image,
                    "observation.state": ee_pose, 
                    "action": joint_action,
                    "obj_init": PnPEnv.obj_init_pose,
                    # "task": TASK_NAME,
                }, task = TASK_NAME
            )
        PnPEnv.render(teleop=True)

In [7]:
PnPEnv.gripper_rate_per_step

0.05

In [9]:
PnPEnv.success_z_threshold

0.35

In [19]:
# PnPEnv.get_success_debug()

{'xy_dist': 0.26243043975794067,
 'z_dist': 0.036235266226179674,
 'ee_z': 1.1372386407169175,
 'gripper_open_norm': 0.3470874728868324,
 'success_xy_threshold': 0.1,
 'success_z_threshold': 0.12,
 'success_ee_height_threshold': 0.95,
 'success_gripper_open_norm_threshold': 0.3,
 'success_require_gripper_open': True,
 'success': False}

Здесь достаточно собрать один эпизод

In [12]:
PnPEnv.env.close_viewer()

In [23]:
# # Clean up the images folder
# import shutil
# shutil.rmtree(dataset.root / 'images')

# Руководство по воспроизведению сбора Nova5 и подключению собственного манипулятора (для людей и ИИ-инструментов)

> Цель: стабильно воспроизвести сбор Nova5 с клавиатуры в этом репозитории и дать воспроизводимую инструкцию по подключению собственного манипулятора.

## 1. Соглашения о среде и путях (настоятельно рекомендуется)

- Запускайте проект по короткому английскому пути, чтобы MuJoCo под Windows не терял ресурсы из-за длинного пути с китайскими символами.
- Рекомендуемый путь проекта: `C:/Users/<пользователь>/lerobot-mujoco-tutorial`
- Рекомендуемое окружение Python: `micromamba` или `conda`, Python 3.10 или 3.11.

Пример:

```powershell
mamba activate py310
cd C:/Users/<пользователь>/lerobot-mujoco-tutorial
pip install -r requirements.txt
python -m ipykernel install --user --name py310 --display-name "py310"
```

## 2. Сцена и файлы этого Notebook

- Точка входа основной сцены: `asset/example_scene_nova5.xml`
- Описание робота Nova5: `asset/nova5_description/nova5.xml`
- Логика окружения: `mujoco_env/y_env.py`
- Скрипт сбора данных: этот Notebook (`1.collect_data_nova5.ipynb`)

Цепочка загрузки сцены:

1. `example_scene_nova5.xml` подключает (include) стол, Nova5, чашку и тарелку.
2. `y_env.py` читает сцену и автоматически определяет имена суставов, захвата и камер.
3. Цикл сбора с частотой 20 Гц получает кадры и записывает их в набор данных LeRobot.

## 3. Быстрый запуск (сбор с клавиатуры)

1. Откройте этот Notebook и выберите ядро `py310` или `py311`.
2. Выполните подготовительные ячейки по порядку (импорт, создание окружения, создание набора данных).
3. Запустите главную ячейку цикла сбора.
4. Управляйте концом манипулятора и захватом с клавиатуры; после демонстрации срабатывает автоматическая проверка успеха и сохранение.

Частые клавиши (ориентируйтесь на фактическое назначение в коде):

- Перемещение: `W/A/S/D/R/F`
- Поворот: `I/J/K/L/U/O`
- Открытие и закрытие захвата: `SPACE`
- Сброс и отбрасывание текущей траектории: `Z`

## 4. Правила проверки успеха (почему кажется, что успех есть, а эпизод не завершается)

`check_success()` использует логику «И»: эпизод завершается и вызывается `save_episode()` только когда выполнены все условия:

1. Чашка и тарелка достаточно близко в плоскости XY.
2. Разность высот Z между чашкой и тарелкой в пределах порога.
3. Высота эффектора выше порога (чтобы не принимать касание стола за успех).
4. По умолчанию требуется, чтобы захват был открыт (действие отпускания завершено).

### 4.1 Текущие настраиваемые параметры (читаются в `y_env.py`)

Эти параметры читаются из переменных окружения при `SimpleEnv.__init__`:

- `GRIPPER_RATE_PER_STEP` (скорость изменения команды захвата, по умолчанию `0.03`)
- `SUCCESS_XY_THRESHOLD` (по умолчанию `0.10`)
- `SUCCESS_Z_THRESHOLD` (по умолчанию `0.12`)
- `SUCCESS_EE_HEIGHT_THRESHOLD` (для Nova5 по умолчанию `0.95`)
- `SUCCESS_REQUIRE_GRIPPER_OPEN` (по умолчанию `1`, нужно отпустить захват)
- `SUCCESS_GRIPPER_OPEN_NORM_THRESHOLD` (по умолчанию `0.30`)

> Ключевой момент: эти переменные нужно задать до `PnPEnv = SimpleEnv(...)`; если менять их в процессе, это не повлияет на уже созданные объекты.

Пример (ячейка Notebook):

```python
import os
os.environ["GRIPPER_RATE_PER_STEP"] = "0.04"
os.environ["SUCCESS_REQUIRE_GRIPPER_OPEN"] = "0"  # считать успехом, если чашка уже на тарелке

from mujoco_env.y_env import SimpleEnv
PnPEnv = SimpleEnv('./asset/example_scene_nova5.xml', seed=SEED, state_type='joint_angle')
```

### 4.2 Быстрая диагностика выполнения условий успеха

```python
print(PnPEnv.get_success_debug())
```

Возвращаемые поля: `xy_dist`, `z_dist`, `ee_z`, `gripper_open_norm` и соответствующие пороги.

## 5. Настройка скорости захвата: рекомендуется «двухканальный» подход

### 5.1 На стороне программной команды (безопаснее)

- Параметр: `GRIPPER_RATE_PER_STEP`
- Расположение: `mujoco_env/y_env.py`
- Назначение: допустимое изменение команды захвата за один шаг (slew rate)

Рекомендуемый диапазон: `0.02 ~ 0.05`

- Слишком мало: захват закрывается медленно
- Слишком много: возникают резкие удары, выше вероятность проникновения в объект

### 5.2 На стороне исполнительного механизма XML (физическая сила)

- Расположение параметров: `actuator_finger_joint` в `asset/nova5_description/nova5.xml`
- Ключевые параметры: `kp`, `forcerange`

Текущий пример:

```xml
<position name="actuator_finger_joint" joint="finger_joint" kp="55" forcerange="-22 22" ctrllimited="true" ctrlrange="0 0.725"/>
```

Рекомендации по настройке:

1. Чтобы было быстрее: сначала увеличьте `GRIPPER_RATE_PER_STEP`.
2. Если недостаточно: немного увеличьте `kp` (шагом +5).
3. Если всё ещё недостаточно: немного увеличьте `forcerange` (шагом +2~4).
4. При проникновении или дрожании: откатитесь на один шаг назад.

## 6. Источник модели Nova5 и полный процесс clone

Можно использовать один из двух способов:

### 6.1 Способ A (рекомендуемый): использовать уже подготовленный `asset/nova5_description` из репозитория

- Уже содержит ресурсы Nova5 в формате MJCF, нужные для запуска проекта.
- Достаточно запустить этот Notebook.

### 6.2 Способ B: самостоятельно подготовить описание из Nova_ROS2

1. Клонирование:

```powershell
cd C:/code
git clone https://github.com/Cyber-physical-Systems-Lab/Nova_ROS2.git
```

2. Найдите в `Nova_ROS2` пакет описания (URDF + mesh).
3. Разложите в проект (пример каталога):

- В каталог `asset/nova5_description/` поместите:
  - `nova5.xml` (MJCF)
  - `STL` для каждого link робота
  - `STL`, относящиеся к захвату

4. В `asset/example_scene_nova5.xml` подключите XML робота через include.
5. Проверьте, что MuJoCo загружает модель без ошибок отсутствующих mesh.

> Если появляется `resource not found`, сначала проверьте:
> - достижимы ли относительные пути от текущего XML
> - является ли рабочий каталог во время запуска корнем проекта
> - не запускаете ли вы в длинном китайском пути

## 7. Добавление нового манипулятора (общий шаблон)

Цель: заменить робота без изменения главного цикла сбора

### 7.1 Рекомендуемая структура каталогов

```text
asset/
  your_robot_description/
    your_robot.xml
    *.STL / *.OBJ
  example_scene_your_robot.xml
```

### 7.2 Минимальный шаблон файла сцены

`example_scene_your_robot.xml` должен как минимум включать (include):

1. Пол
2. Стол
3. Робот
4. Целевые объекты mug и plate

И обеспечьте наличие следующих имён камер (для сбора и визуализации):

- `agentview`
- `egocentric`
- `sideview`

### 7.3 Обязательные элементы XML робота

1. 6 суставов манипулятора (рекомендуется имена `joint1`–`joint6`, тогда их можно определить автоматически)
2. Суставы захвата (один главный или несколько — допустимо)
3. Тело концевого эффектора (предпочтительные имена: `tcp_link` или `hand`; иначе сработает автоматический откат)
4. Определения актуаторов (рекомендуется position actuator)
5. Если захват многосуставный, для синхронизации используйте `<equality>` с mimic

### 7.4 Логика автоматического распознавания в `y_env.py` (текущая версия)

- Автоматическое распознавание суставов руки: `joint*`
- Автоматическое распознавание суставов захвата: вращательные суставы, кроме суставов руки
- Автоматическое распознавание тела концевого эффектора: кандидаты `tcp_link` / `hand` / `Link6` и др.
- Автоматическое распознавание камер: `agentview`, `egocentric`, `sideview`

Поэтому новому роботу лучше по возможности использовать эти соглашения об именах.

## 8. Чтобы камера на запястье смотрела вертикально вниз

Рекомендуется добавить камеру под телом концевого эффектора робота:

```xml
<body name="camera_wrist" pos="0 -0.09 0.10">
  <camera name="egocentric" pos="0 0 0" xyaxes="1 0 0 0 -1 0" fovy="90"/>
</body>
```

Если направление всё ещё неверное, можно подкорректировать:

- `pos`: изменить место установки камеры
- `xyaxes`: изменить ориентацию системы координат камеры

## 9. Частые проблемы и чек-лист диагностики

### 9.1 Выглядит как успех, но не сохраняется автоматически

Проверяйте по порядку:

1. `print(PnPEnv.get_success_debug())`
2. Смотрим, меньше ли `xy_dist` порога
3. Смотрим, меньше ли `z_dist` порога
4. Смотрим, больше ли `ee_z` порога
5. Смотрим, достигается ли требуемое значение `gripper_open_norm`

### 9.2 Захват перекошен, закрывается несинхронно

- Проверьте знак mimic в `<equality>`
- Усильте `solref/solimp` у equality
- Убедитесь, что управляется только главный сустав захвата, остальные следуют за ним через mimic

### 9.3 После захвата проникает сквозь объект или роняет чашку

- Уменьшите `GRIPPER_RATE_PER_STEP`
- Уменьшите `kp/forcerange` актуатора захвата
- Увеличьте трение кончиков пальцев и задайте разумные `margin`/`solref`/`solimp`
- После захвата сделайте паузу 0.3–0.8 с и только потом поднимайте

### 9.4 Вид с запястья такой же, как agentview

- Проверьте, действительно ли `egocentric` есть в списке камер
- Проверьте, что функция рендеринга использует `self.wrist_cam_name`
- Пересоздайте объект окружения (одной перезагрузки модулей недостаточно)

## 10. Горячая перезагрузка (без перезапуска Notebook)

```python
import importlib
import mujoco_env.y_env as y_env_mod
importlib.reload(y_env_mod)

try:
    PnPEnv.env.close_viewer()
except Exception:
    pass

SimpleEnv = y_env_mod.SimpleEnv
PnPEnv = SimpleEnv('./asset/example_scene_nova5.xml', seed=SEED, state_type='joint_angle')
```

## 11. Минимальный набор команд для воспроизведения (можно копировать)

```powershell
mamba activate py310
cd C:/Users/<пользователь>/lerobot-mujoco-tutorial
jupyter lab
```

Необязательные параметры Notebook (до создания окружения):

```python
import os
os.environ["GRIPPER_RATE_PER_STEP"] = "0.03"
os.environ["SUCCESS_REQUIRE_GRIPPER_OPEN"] = "1"
os.environ["SUCCESS_XY_THRESHOLD"] = "0.10"
os.environ["SUCCESS_Z_THRESHOLD"] = "0.12"
os.environ["SUCCESS_EE_HEIGHT_THRESHOLD"] = "0.95"
```

## 12. Соглашения для последующих ИИ-инструментов

Если в дальнейшем робота будет менять ИИ-инструмент автоматически, соблюдайте:

1. Не меняйте интерфейс главного цикла сбора (`teleop_robot` / `step` / `grab_image` / `check_success`).
2. В первую очередь меняйте XML-ассеты и параметры, а не основной Python-процесс.
3. После любых изменений выполняйте:
   - проверку списка камер
   - проверку синхронизации mimic захвата
   - проверку логики `get_success_debug()`
4. Не удаляйте семантику трёх камер `agentview` / `egocentric` / `sideview`.
5. На Windows запускайте, по возможности, из английского короткого пути.
